# Database Setup
This jupyter notebook walks through the setup and some queries on the MAT125_data database. It covers database initialization, creating tables from CSVs, and schema correction. 

### Getting SQLite
1. install sqlite3 from [the SQLite website](https://www.sqlite.org/download.html). You can verify installation using `sqlite3 --version`
* Mac may have this pre-installed
* Linux users: sudo apt/yum/pacman install sqlite. Choose your apropriate package manager
2. create a database using `sqlite3 database_name.db`. Here, I use `MAT125_data.db`

## Loading our CSV files into our database
Start by loading some libraries

In [11]:
import sqlite3 as sql
import pandas as pd
# import re # For regex expressions if looping files

Then, we grab our filenames. If there are many files, we could read them into the database using a loop and a list of files. Since we are working with 2 here, I will do them one by one. The data in the `canvas.csv` is a special case.

In [12]:
DB_NAME = "MAT125_data.db"

# Load files to put into database
pearson_file = "Cleaned_For_DataSci/pearson.csv"
SIS_file = "Cleaned_For_DataSci/SIS.csv"

# Will need to treat canvas specially because
# SQLite only allows for 2,000 predictors. This has 5,919
canvas = "Cleaned_For_DataSci/canvas.csv"

If we wanted to load all files in using a loop, we could use the following code:


However, this code shows CSVs loaded one by one. It mirrors the above code closely. Using `with` makes the following code run within the context of the open connection and makes sure we close the connection properly at the end.

`pd.read_sql_query()` is a function which takes an SQL query and a connection to the relevant database and returns the requested information. It does not change the stored data or database's structure.

In [13]:
# Pearson data
# Read csv into a pandas DataFrame 
df = pd.read_csv(pearson_file)

# Open a connection to our database
with sql.connect(DB_NAME) as connection:
    # Load dataframe into database
        # Pearson is the table name
    df.to_sql("Pearson", 
                connection, 
                if_exists='replace', 
                index=False)

    # If we want to see how big our table was when loaded in
    # and its structure
    print("Pearson")
    print(f"\tRows: {df.shape[0]}, Columns: {df.shape[1]}")
    print(pd.read_sql_query("PRAGMA table_info(Pearson);", connection))


Pearson
	Rows: 218546, Columns: 12
    cid                 name     type  notnull dflt_value  pk
0     0           Identifier     REAL        0       None   0
1     1               Status     TEXT        0       None   0
2     2               Course     TEXT        0       None   0
3     3          Course.Code     TEXT        0       None   0
4     4     Assignment.Title     TEXT        0       None   0
5     5  Assignment.Category     TEXT        0       None   0
6     6       Assignment.Tag     TEXT        0       None   0
7     7           Time.Spent     TEXT        0       None   0
8     8                Score     REAL        0       None   0
9     9             semester     TEXT        0       None   0
10   10         section_code     TEXT        0       None   0
11   11                 Term  INTEGER        0       None   0


The same can be done with the SIS file.

In [14]:
# Read csv into a pandas DataFrame 
df = pd.read_csv(SIS_file)

# Open a connection to our database
with sql.connect(DB_NAME) as connection:
    # Load dataframe into database
        # Pearson is the table name
    df.to_sql("SIS", 
                connection, 
                if_exists='replace', 
                index=False)

    # If we want to see how big our table was when loaded in
    # and its structure
    print("SIS")
    print(f"\tRows: {df.shape[0]}, Columns: {df.shape[1]}")
    print(pd.read_sql_query("PRAGMA table_info(SIS);", connection))


SIS
	Rows: 2320, Columns: 45
    cid                          name     type  notnull dflt_value  pk
0     0                    Identifier  INTEGER        0       None   0
1     1                          Term     REAL        0       None   0
2     2                         Class     TEXT        0       None   0
3     3                     Class.Nbr     REAL        0       None   0
4     4                       Section     REAL        0       None   0
5     5   Primary.Instruction.Section     REAL        0       None   0
6     6             Class.Description     TEXT        0       None   0
7     7                Official.Grade     TEXT        0       None   0
8     8                     Status.Cd     TEXT        0       None   0
9     9                 Status.Reason     TEXT        0       None   0
10   10                Enrolled.Hours     REAL        0       None   0
11   11                      Add.Date     TEXT        0       None   0
12   12               IPEDS.Ethnicity     TEXT  

There is one issue though. Pandas gave datatypes to each column in Pearson, but not in SIS. Our database does not know what type of data each field contains. We can set each field manually using the `dtype` parameter when calling `df.to_sql()`.

We can define the `dtype` mapping as a python dictionary with column names as keys and data types as values. Data types are listed in all caps.

In [15]:
sis_dtype = {
    # Identifiers / keys
    "Identifier": "INTEGER",
    "Class.Nbr": "INTEGER",
    "section_code": "TEXT",

    # Term / enrollment timing
    "Term": "INTEGER",
    "Add.Date": "TEXT",              # keep as TEXT (ISO dates play nicest in SQLite)
    "First.Term.Enrolled.Cd.Ugrd": "INTEGER",
    "First.Term.Enrolled.Cd.Grad": "INTEGER",
    "Last.Term.Enrolled": "INTEGER",

    # Course / class info
    "Class": "TEXT",
    "Section": "TEXT",
    "Primary.Instruction.Section": "TEXT",
    "Class.Description": "TEXT",
    "Class.Campus.Cd": "TEXT",
    "Class.Campus": "TEXT",
    "Instruction.Mode": "TEXT",

    # Enrollment / academic status
    "Official.Grade": "TEXT",
    "Status.Cd": "TEXT",
    "Status.Reason": "TEXT",
    "Enrolled.Hours": "REAL",

    # Demographics
    "IPEDS.Ethnicity": "TEXT",
    "Sex": "TEXT",
    "Age": "INTEGER",
    "1st.Gen.College.Std.Flag": "INTEGER",
    "X1st.Gen.College.Std.Flag": "INTEGER",

    # Location (mailing + home)
    "Mail.City": "TEXT",
    "Mail.State.Cd": "TEXT",
    "Mail.Country": "TEXT",
    "Home.City": "TEXT",
    "Home.State.Cd": "TEXT",
    "Home.County": "TEXT",
    "Home.Country": "TEXT",

    # Academic standing
    "Academic.Level.Begin.of.Term": "TEXT",
    "Cum.GPA": "REAL",
    "Total.Cum.Units": "REAL",
    "Total.NAU.Units": "REAL",

    # Academic structure
    "Primary.Academic.Program": "TEXT",
    "Primary.Academic.Plan": "TEXT",
    "Sub.Plan": "TEXT",
    "Student.Campus": "TEXT",
    "College.Cd": "TEXT",
    "College": "TEXT",
    "Division.Cd": "TEXT",
    "Division": "TEXT",
    "Unit.Cd": "TEXT",
    "Unit": "TEXT",
}

There are two methods we can use to fix the missing data type information; both require that we drop the entire table, so this is good to do early when importing CSVs to a database.

### Option 1 : Re-Read from CSV
1. Drop existing table
2. Re-read data from the csv into a pandas DataFrame
3. Load the csv to the database, now using the `dtype` mapping

In [16]:
# Open a connection to the database 
with sql.connect(DB_NAME) as connection:

# Create a cursor to execute commands
    cursor = connection.cursor()
    # drop existing SIS table 
        # sqlite does not allow changing column times in place, so
        # we will need to re-build the table
    connection.execute("DROP TABLE IF EXISTS SIS;")

    # Re-read in SIS CSV
    sis_df = pd.read_csv("Cleaned_For_DataSci/SIS.csv")

    # Re-create the SIS table, now with the datatype schema
    sis_df.to_sql(
        "SIS",
        connection,
        if_exists = 'replace',
        index = False,
        dtype = sis_dtype
    )

    # Check that this got fixed
    print(pd.read_sql_query("PRAGMA table_info(SIS);", connection))


    cid                          name     type  notnull dflt_value  pk
0     0                    Identifier  INTEGER        0       None   0
1     1                          Term  INTEGER        0       None   0
2     2                         Class     TEXT        0       None   0
3     3                     Class.Nbr  INTEGER        0       None   0
4     4                       Section     TEXT        0       None   0
5     5   Primary.Instruction.Section     TEXT        0       None   0
6     6             Class.Description     TEXT        0       None   0
7     7                Official.Grade     TEXT        0       None   0
8     8                     Status.Cd     TEXT        0       None   0
9     9                 Status.Reason     TEXT        0       None   0
10   10                Enrolled.Hours     REAL        0       None   0
11   11                      Add.Date     TEXT        0       None   0
12   12               IPEDS.Ethnicity     TEXT        0       None   0
13   1

We see how any real changes to our database, not just pulling information, will require a **cursor** to run the relevant command through `conection.execute()`.

### Option 2 : Read from database

This is helpful when applying a data type schema to an existing table when you do not already have the data stored in a separate `.csv` file.

1. Grab all the data from the table in the existing database and store it in a pandas DataFrame.
2. Drop the table from the database. This is only safe to do because we saved the data in a DataFrame in step 1 BEFORE dropping the table.
3. Re-load in the DataFrame to the database, now with the schema.

In [17]:
# Open a connection to the database 
with sql.connect(DB_NAME) as connection:

# Create a cursor to execute commands
    cursor = connection.cursor()
   
    # Select all entries from SIS and save into pd.DataFrame sis_df
    sis_df = pd.read_sql_query("SELECT * FROM SIS;", connection)

    # Now that we have saved our data, we can drop the table
    connection.execute("DROP TABLE SIS;")

    sis_df.to_sql(
        "SIS",
        connection,
        if_exists="replace",
        index=False,
        dtype=sis_dtype
    )

Though these fixes work, it might be clear that loading a csv to a database initially with the full and correct `dtype` mapping is good practice.

Let's peek at our tables:

In [ ]:
with sql.connect(DB_NAME) as connection:
    tables = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table';",
            connection
        )

    print("\nTables in Database:\n")
    print(tables)


Tables in Database:
,       name
0  Pearson
1      SIS


We can also empty our database using DROP TABLE.

In [19]:
with sql.connect(DB_NAME) as connection:
    cursor = connection.cursor()
    connection.execute("DROP TABLE Pearson;")
    connection.execute("DROP TABLE SIS;")

    # Now we should have an empty database with no tables
    tables = pd.read_sql_query(
            "SELECT name FROM sqlite_master WHERE type='table';",
            connection
        )

    print("\nTables in Database:\n")
    print(tables)


Tables in Database:

Empty DataFrame
Columns: [name]
Index: []
